# NeMo Curator - Simple Wikipedia Download & Extract

Run this notebook inside the NeMo Curator Docker container. Everything runs in-container - no local installation needed.

In [2]:
from nemo_curator.core.client import RayClient
from nemo_curator.pipeline.pipeline import Pipeline
from nemo_curator.stages.text.download.wikipedia.stage import WikipediaDownloadExtractStage
from nemo_curator.stages.text.io.writer.jsonl import JsonlWriter
import os
from pathlib import Path
print('OK')

OK


## Configuration

In [ ]:
# Paths (mounted volumes - files persist on your host)
DOWNLOAD_DIR = "/workspace/wikipedia/data"
OUTPUT_DIR = "/workspace/wikipedia/output"

# Pipeline settings
LANGUAGE = "simple"       # Simple Wikipedia
URL_LIMIT = 2             # Limit dump files for testing
RECORD_LIMIT = 10       # Limit articles per file for testing
print('OK')

OK


## Start Ray Cluster

In [4]:
print("Starting local Ray cluster...")
ray_client = RayClient()
ray_client.start()
print("Ray cluster started!")

2026-04-14 17:29:11.978 | WARNING  | nemo_curator.core.client:start:106 - No monitoring services are running. Please run the `start_prometheus_grafana.py` script from nemo_curator/metrics folder to setup monitoring services separately.
2026-04-14 17:29:12.084 | INFO     | nemo_curator.core.utils:init_cluster:135 - Ray start command: ray start --head --node-ip-address 172.17.0.2 --port 6379 --metrics-export-port 8080 --dashboard-host 127.0.0.1 --dashboard-port 8265 --ray-client-server-port 10001 --temp-dir /tmp/ray --disable-usage-stats --block


Starting local Ray cluster...
Ray cluster started!


2026-04-14 17:29:16,489	WARNING services.py:2137 -- WARNING: The object store is using /tmp instead of /dev/shm because /dev/shm has only 67108864 bytes available. This will harm performance! You may be able to free up space by deleting files in /dev/shm. If you are inside a Docker container, you can increase /dev/shm size by passing '--shm-size=10.24gb' to 'docker run' (or add it to the run_options list in a Ray cluster config). Make sure to set this to more than 30% of available RAM.


2026-04-14 17:29:13,707	INFO usage_lib.py:447 -- Usage stats collection is disabled.
2026-04-14 17:29:13,707	INFO scripts.py:919 -- Local node IP: 172.17.0.2
2026-04-14 17:29:16,601	SUCC scripts.py:963 -- --------------------
2026-04-14 17:29:16,602	SUCC scripts.py:964 -- Ray runtime started.
2026-04-14 17:29:16,602	SUCC scripts.py:965 -- --------------------
2026-04-14 17:29:16,602	INFO scripts.py:967 -- Next steps
2026-04-14 17:29:16,602	INFO scripts.py:970 -- To add another node to this Ray cluster, run
2026-04-14 17:29:16,602	INFO scripts.py:973 --   ray start --address='172.17.0.2:6379'
2026-04-14 17:29:16,602	INFO scripts.py:982 -- To connect to this Ray cluster:
2026-04-14 17:29:16,602	INFO scripts.py:984 -- import ray
2026-04-14 17:29:16,602	INFO scripts.py:985 -- ray.init(_node_ip_address='172.17.0.2')
2026-04-14 17:29:16,602	INFO scripts.py:997 -- To submit a Ray job using the Ray Jobs CLI:
2026-04-14 17:29:16,602	INFO scripts.py:998 --   RAY_API_SERVER_ADDRESS='http://127.0.

## Run Wikipedia Pipeline

In [5]:
stage = WikipediaDownloadExtractStage(
    language=LANGUAGE,
    download_dir=DOWNLOAD_DIR,
    url_limit=URL_LIMIT,
    record_limit=RECORD_LIMIT,
    verbose=True,
)

pipeline = Pipeline("wikipedia_simple")
pipeline.add_stage(stage)

print("Running download and extract pipeline...")
results = pipeline.run()
print(f"Pipeline completed. Got {len(results)} result batches.")

2026-04-14 17:29:23.414 | INFO     | nemo_curator.pipeline.pipeline:add_stage:61 - Added stage 'wikipedia_simple_pipeline' to pipeline 'wikipedia_simple'
2026-04-14 17:29:23.415 | INFO     | nemo_curator.pipeline.pipeline:build:70 - Planning pipeline: wikipedia_simple
2026-04-14 17:29:23.416 | INFO     | nemo_curator.pipeline.pipeline:_decompose_stages:106 - Decomposing composite stage: wikipedia_simple_pipeline
2026-04-14 17:29:23.417 | INFO     | nemo_curator.pipeline.pipeline:_decompose_stages:120 - Expanded 'wikipedia_simple_pipeline' into 4 execution stages


Running download and extract pipeline...


2026-04-14 17:29:23.651 | INFO     | nemo_curator.backends.xenna.executor:execute:135 - Execution mode: STREAMING
2026-04-14 17:29:23,651	INFO worker.py:1696 -- Using address 172.17.0.2:6379 set in the environment variable RAY_ADDRESS
2026-04-14 17:29:23,654	INFO worker.py:1837 -- Connecting to existing Ray cluster at address: 172.17.0.2:6379...
2026-04-14 17:29:23,663	INFO worker.py:2014 -- Connected to Ray cluster. View the dashboard at http://127.0.0.1:8265 
/opt/venv/lib/python3.12/site-packages/ray/_private/worker.py:2062: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(
2026-04-14 17:29:25.378 | INFO     | nemo_curator.backends.xenna.adapter:required_resources:43 - Resources: Resources(cpus=0.5, gpu_memory_gb=0.0, nvdecs=0, nvencs=0, entire_gpu=False, gpus=0.0)
202

Pipeline completed. Got 1 result batches.


## Inspect Results

In [6]:
if results:
    print(f"Batch 0 shape: {results[0].data.shape}")
    display(results[0].data.head())

Batch 0 shape: (1000, 7)


,text,title,id,url,language,source_id,file_name
0,April (Apr.) is the fourth month of the year i...,April,1,https://simple.wikipedia.org/wiki/April,simple,simplewiki-20260401-simplewiki-20260401-pages-...,simplewiki-20260401-simplewiki-20260401-pages-...
1,August (Aug.) is the eighth month of the year ...,August,2,https://simple.wikipedia.org/wiki/August,simple,simplewiki-20260401-simplewiki-20260401-pages-...,simplewiki-20260401-simplewiki-20260401-pages-...
2,Art is a creative activity. It produces a prod...,Art,6,https://simple.wikipedia.org/wiki/Art,simple,simplewiki-20260401-simplewiki-20260401-pages-...,simplewiki-20260401-simplewiki-20260401-pages-...
3,A is the first letter of the English alphabet....,A,8,https://simple.wikipedia.org/wiki/A,simple,simplewiki-20260401-simplewiki-20260401-pages-...,simplewiki-20260401-simplewiki-20260401-pages-...
4,Air is the Earth's atmosphere. Air is a mixtur...,Air,9,https://simple.wikipedia.org/wiki/Air,simple,simplewiki-20260401-simplewiki-20260401-pages-...,simplewiki-20260401-simplewiki-20260401-pages-...


## Save to JSONL

In [7]:
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

jsonl_writer = JsonlWriter(path=OUTPUT_DIR, write_kwargs={"force_ascii": False})

for idx, result in enumerate(results):
    output_path = os.path.join(OUTPUT_DIR, f"{idx}.jsonl")
    jsonl_writer.write_data(result, output_path)
    print(f"Written: {output_path}")

print(f"\nJSONL files saved to: {OUTPUT_DIR}")

Written: /workspace/wikipedia/output/0.jsonl

JSONL files saved to: /workspace/wikipedia/output


## Cleanup

In [8]:
try:
    ray_client.stop()
    print("Ray cluster stopped.")
except Exception as e:
    print(f"Warning: Error stopping Ray client: {e}")

2026-04-14 17:31:12.031 | INFO     | nemo_curator.core.client:stop:181 - NeMo Curator has stopped the Ray cluster it started by killing the Ray GCS process. It is advised to wait for a few seconds before running any Ray commands to ensure Ray can cleanup other processes.If you are seeing any Ray commands like `ray status` failing, please ensure /tmp/ray/ray_current_cluster has correct information.


Ray cluster stopped.
